# 07 — Projeções comparativas 2D e 3D

Reúne, no mesmo gráfico, as fronteiras obtidas por todos os métodos válidos e a fronteira de Pareto verdadeira para os quatro cenários selecionados para o texto: `m4_medium`, `m6_low`, `m6_medium` e `m12_high`. As cores são idênticas às adotadas no notebook 06; formas distintas tornam a comparação também legível sem depender apenas da cor.

As sementes representativas continuam sendo as escolhidas no notebook 06 pela proximidade à IGD mediana. As projeções 3D usam projeção ortográfica e vista isométrica (`elevação = 35,264°`, `azimute = -45°`). Nenhum otimizador é reexecutado.

In [ ]:
from pathlib import Path
import json, os, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

def project_root(start=Path.cwd()):
    p=start.resolve()
    for candidate in (p,*p.parents):
        if (candidate/'configs'/'smoke.json').exists(): return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada')

ROOT=project_root(); MODE=os.environ.get('CNBI_MODE','SMOKE').upper(); ALPHA=2**0.75
OUT=ROOT/'results'/'figures'/'selected_combined_projections'; OUT.mkdir(parents=True,exist_ok=True)
OVERLAYS=ROOT/'results'/'figures'/'method_front_overlays'
METHOD_ORDER=['NBI','CNBI','VRF-NBI','NSGA-III','MOEA/D']
METHOD_COLOR={'NBI':'#0072b2','CNBI':'#d55e00','VRF-NBI':'#009e73','NSGA-III':'#cc79a7','MOEA/D':'#e69f00'}
METHOD_MARKER={'NBI':'P','CNBI':'o','VRF-NBI':'^','NSGA-III':'s','MOEA/D':'D'}
SELECTIONS={
    'm4_medium': {'pairs':[(1,2),(3,4)], 'triples':[(1,2,3),(2,3,4)]},
    'm6_low': {'pairs':[(1,2),(3,4),(5,6)], 'triples':[(1,2,3),(4,5,6)]},
    'm6_medium': {'pairs':[(1,2),(3,4),(5,6)], 'triples':[(1,2,3),(4,5,6)]},
    'm12_high': {'pairs':[(1,2),(6,7),(11,12)], 'triples':[(1,2,3),(5,6,7),(10,11,12)]},
}
ISO_ELEV=35.264; ISO_AZIM=-45.0
plt.rcParams.update({'font.family':'DejaVu Serif','font.size':9,'axes.titlesize':10,'axes.labelsize':9,'legend.fontsize':8,'figure.titlesize':13,'savefig.facecolor':'white','axes.facecolor':'white'})

def slug(text): return re.sub(r'[^a-z0-9]+','-',text.lower()).strip('-')
def sample_indices(n,limit,seed):
    if n<=limit: return np.arange(n)
    return np.sort(np.random.default_rng(seed).choice(n,size=limit,replace=False))
def relative(path): return path.relative_to(ROOT).as_posix()


In [ ]:
def method_front(record,anchors):
    data=np.load(ROOT/record.checkpoint,allow_pickle=False); X=np.asarray(data['X'],float); success=np.asarray(data['success'],bool); data.close()
    feasible=np.sum(X*X,axis=1)<=ALPHA**2+1e-8; X=X[success&feasible]
    F=np.sum((X[:,None,:]-anchors[None,:,:])**2,axis=2)
    keep=NonDominatedSorting().do(F,only_non_dominated_front=True); F=F[keep]
    assert len(F)==int(record.expected_n), f'Cardinalidade divergente: {record.scenario}/{record.method}/{record.seed}'
    return F

def load_scenario(scenario,selected):
    scenario_data=np.load(ROOT/'data'/'generated'/f'{scenario}_scenario.npz'); anchors=np.asarray(scenario_data['anchors'],float); scenario_data.close()
    reference=np.load(ROOT/'data'/'reference_fronts'/f'{scenario}_pareto_reference.npz'); Fref=np.asarray(reference['F'],float); reference.close()
    records=selected[selected.scenario.eq(scenario)].sort_values('method')
    fronts={row.method:method_front(row,anchors) for row in records.itertuples(index=False)}
    methods=[method for method in METHOD_ORDER if method in fronts]
    assert methods and ('NBI' in methods)==scenario.startswith('m4_')
    return anchors,Fref,fronts,methods

def anchor_objectives(anchors): return np.sum((anchors[:,None,:]-anchors[None,:,:])**2,axis=2)

def handles(methods):
    items=[Line2D([0],[0],marker='o',linestyle='none',markerfacecolor='#a7adb4',markeredgecolor='none',alpha=.55,label='fronteira verdadeira')]
    items.extend(Line2D([0],[0],marker=METHOD_MARKER[m],linestyle='none',markerfacecolor=METHOD_COLOR[m],markeredgecolor='white',markeredgewidth=.4,markersize=6,label=m) for m in methods)
    items.append(Line2D([0],[0],marker='*',linestyle='none',markerfacecolor='white',markeredgecolor='.1',markersize=8,label='ótimos individuais verdadeiros'))
    return items

def draw_2d(ax,pair,Fref,fronts,methods,Fanchors,seed):
    i,j=(pair[0]-1,pair[1]-1); idx=sample_indices(len(Fref),6000,1907+seed+i*31+j); true=Fref[idx]
    ax.scatter(true[:,i],true[:,j],s=4,c='#a7adb4',alpha=.18,linewidths=0,rasterized=True)
    for method in sorted(methods,key=lambda m:len(fronts[m]),reverse=True):
        F=fronts[method]; size=12 if len(F)>1000 else 16 if len(F)>300 else 23 if len(F)>100 else 32; opacity=.48 if len(F)>1000 else .62 if len(F)>300 else .78
        ax.scatter(F[:,i],F[:,j],s=size,c=METHOD_COLOR[method],marker=METHOD_MARKER[method],alpha=opacity,edgecolors='white',linewidths=.3,rasterized=True)
    ax.scatter(Fanchors[:,i],Fanchors[:,j],marker='*',s=55,c='white',edgecolors='.1',linewidths=.7,zorder=8)
    ax.set(xlabel=rf'$f_{{{i+1}}}$',ylabel=rf'$f_{{{j+1}}}$',title=rf'Projeção $f_{{{i+1}}}\times f_{{{j+1}}}$'); ax.grid(alpha=.15)

def draw_3d(ax,triple,Fref,fronts,methods,Fanchors,seed):
    i,j,k=(triple[0]-1,triple[1]-1,triple[2]-1); idx=sample_indices(len(Fref),4500,2909+seed+i*31+j*17+k); true=Fref[idx]
    ax.scatter(true[:,i],true[:,j],true[:,k],s=3,c='#a7adb4',alpha=.12,linewidths=0,depthshade=False,rasterized=True)
    for method in sorted(methods,key=lambda m:len(fronts[m]),reverse=True):
        F=fronts[method]; size=10 if len(F)>1000 else 15 if len(F)>300 else 22 if len(F)>100 else 32; opacity=.50 if len(F)>1000 else .65 if len(F)>300 else .82
        ax.scatter(F[:,i],F[:,j],F[:,k],s=size,c=METHOD_COLOR[method],marker=METHOD_MARKER[method],alpha=opacity,edgecolors='white',linewidths=.3,depthshade=False,rasterized=True)
    ax.scatter(Fanchors[:,i],Fanchors[:,j],Fanchors[:,k],marker='*',s=58,c='white',edgecolors='.1',linewidths=.7,depthshade=False)
    ax.set(xlabel=rf'$f_{{{i+1}}}$',ylabel=rf'$f_{{{j+1}}}$',title=rf'Projeção $(f_{{{i+1}}},f_{{{j+1}}},f_{{{k+1}}})$'); ax.set_zlabel('')
    ax.text2D(.91,.52,rf'$f_{{{k+1}}}$',transform=ax.transAxes,rotation=90,va='center',ha='center')
    ax.view_init(elev=ISO_ELEV,azim=ISO_AZIM); ax.set_proj_type('ortho'); ax.set_box_aspect((1,1,1)); ax.grid(alpha=.15)
    for axis in (ax.xaxis,ax.yaxis,ax.zaxis): axis.pane.set_facecolor((1,1,1,0))


In [ ]:
def save_individual_2d(scenario,pair,Fref,fronts,methods,Fanchors,out_dir,seed):
    fig,ax=plt.subplots(figsize=(7.2,6.2),layout='constrained'); draw_2d(ax,pair,Fref,fronts,methods,Fanchors,seed)
    fig.legend(handles=handles(methods),loc='outside lower center',ncol=min(4,len(methods)+2),frameon=False)
    fig.suptitle(f'{scenario}: todos os métodos sobre a fronteira verdadeira')
    stem=f'{scenario}_all_methods_f{pair[0]}_f{pair[1]}_2d'; png=out_dir/f'{stem}.png'; pdf=out_dir/f'{stem}.pdf'
    fig.savefig(png,dpi=300,bbox_inches='tight'); fig.savefig(pdf,dpi=300,bbox_inches='tight'); plt.close(fig); return png,pdf

def save_individual_3d(scenario,triple,Fref,fronts,methods,Fanchors,out_dir,seed):
    fig=plt.figure(figsize=(7.5,6.8),layout='constrained'); ax=fig.add_subplot(111,projection='3d'); draw_3d(ax,triple,Fref,fronts,methods,Fanchors,seed)
    fig.legend(handles=handles(methods),loc='outside lower center',ncol=min(4,len(methods)+2),frameon=False)
    fig.suptitle(f'{scenario}: todos os métodos — vista isométrica')
    stem=f'{scenario}_all_methods_f{triple[0]}_f{triple[1]}_f{triple[2]}_3d_isometric'; png=out_dir/f'{stem}.png'; pdf=out_dir/f'{stem}.pdf'
    fig.savefig(png,dpi=300,bbox_inches='tight'); fig.savefig(pdf,dpi=300,bbox_inches='tight'); plt.close(fig); return png,pdf

def save_plate(scenario,projections,dimension,Fref,fronts,methods,Fanchors,out_dir,seed):
    n=len(projections); fig=plt.figure(figsize=(6.1*n,6.2 if dimension=='2D' else 6.8),layout='constrained')
    for pos,projection in enumerate(projections,1):
        ax=fig.add_subplot(1,n,pos,projection=None if dimension=='2D' else '3d')
        (draw_2d if dimension=='2D' else draw_3d)(ax,projection,Fref,fronts,methods,Fanchors,seed+pos)
    fig.legend(handles=handles(methods),loc='outside lower center',ncol=len(methods)+2,frameon=False)
    suffix='2d' if dimension=='2D' else '3d_isometric'; fig.suptitle(f'{scenario}: todos os métodos sobre a fronteira verdadeira' + ('' if dimension=='2D' else ' — vistas isométricas'))
    png=out_dir/f'{scenario}_all_methods_plate_{suffix}.png'; pdf=out_dir/f'{scenario}_all_methods_plate_{suffix}.pdf'
    fig.savefig(png,dpi=300,bbox_inches='tight'); fig.savefig(pdf,dpi=300,bbox_inches='tight'); plt.close(fig); return png,pdf


In [ ]:
selected=pd.read_csv(OVERLAYS/f'{MODE.lower()}_representative_seeds.csv'); rows=[]
for scenario,spec in SELECTIONS.items():
    if scenario not in set(selected.scenario): continue
    scenario_dir=OUT/scenario; dir2d=scenario_dir/'2d'; dir3d=scenario_dir/'3d_isometric'; dir2d.mkdir(parents=True,exist_ok=True); dir3d.mkdir(parents=True,exist_ok=True)
    anchors,Fref,fronts,methods=load_scenario(scenario,selected); Fanchors=anchor_objectives(anchors); method_seeds={m:int(selected[(selected.scenario==scenario)&(selected.method==m)].seed.iloc[0]) for m in methods}; seed=sum(method_seeds.values())
    plate2png,plate2pdf=save_plate(scenario,spec['pairs'],'2D',Fref,fronts,methods,Fanchors,scenario_dir,seed)
    plate3png,plate3pdf=save_plate(scenario,spec['triples'],'3D',Fref,fronts,methods,Fanchors,scenario_dir,seed)
    for pair in spec['pairs']:
        png,pdf=save_individual_2d(scenario,pair,Fref,fronts,methods,Fanchors,dir2d,seed)
        rows.append({'scenario':scenario,'dimension':'2D','projection':'-'.join(map(str,pair)),'objective_i':pair[0],'objective_j':pair[1],'objective_k':np.nan,'methods':'|'.join(methods),'method_seeds':json.dumps(method_seeds,sort_keys=True),'n_methods':len(methods),'png':relative(png),'pdf':relative(pdf),'plate_png':relative(plate2png),'plate_pdf':relative(plate2pdf),'view_elev':np.nan,'view_azim':np.nan})
    for triple in spec['triples']:
        png,pdf=save_individual_3d(scenario,triple,Fref,fronts,methods,Fanchors,dir3d,seed)
        rows.append({'scenario':scenario,'dimension':'3D','projection':'-'.join(map(str,triple)),'objective_i':triple[0],'objective_j':triple[1],'objective_k':triple[2],'methods':'|'.join(methods),'method_seeds':json.dumps(method_seeds,sort_keys=True),'n_methods':len(methods),'png':relative(png),'pdf':relative(pdf),'plate_png':relative(plate3png),'plate_pdf':relative(plate3pdf),'view_elev':ISO_ELEV,'view_azim':ISO_AZIM})
manifest=pd.DataFrame(rows); manifest.to_csv(OUT/f'{MODE.lower()}_selected_combined_projection_manifest.csv',index=False)
metadata={'mode':MODE,'scenarios':list(SELECTIONS),'method_colors':METHOD_COLOR,'method_markers':METHOD_MARKER,'selection_rule':'representative seeds inherited from notebook 06','reference_source':'direct true Pareto reference from data/reference_fronts','two_dimensional_projections':{k:v['pairs'] for k,v in SELECTIONS.items()},'three_dimensional_projections':{k:v['triples'] for k,v in SELECTIONS.items()},'three_dimensional_view':{'projection':'orthographic','elevation_degrees':ISO_ELEV,'azimuth_degrees':ISO_AZIM},'max_reference_points_2d':6000,'max_reference_points_3d':4500}
(OUT/f'{MODE.lower()}_selected_combined_projection_metadata.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf-8')
print(f'Projeções combinadas {MODE}: {len(manifest)} projeções em {manifest.scenario.nunique()} cenários.')
